# ML-04 — Search Intelligence Data Contract

This notebook is the **written, query-backed contract** for the content-decline prediction task.
Every claim is followed by the code that verifies it.

> Skills loaded: `writing-data-contracts/SKILL.md`, `flyrank/flyrank-data/SKILL.md`

---

## Setup — authenticate and register tables

In [ ]:
# Cell 0 — install (run once)
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass, duckdb

# Colab Secrets (🔑 panel) injects HF_TOKEN; getpass is the safe fallback.
# NEVER paste a token directly into a code cell — this repo is public.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Smoke-test: reads only Parquet footer metadata — finishes in seconds
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Q1 — Unit of analysis:**
One row equals one content item's aggregated performance over a trailing monthly window, so for a partition like `month=2026-03` the grain is `content_hash_id × month`.

**Q3 — Time window (label):**
The label compares each content item's total impressions in month T+1 against month T, with all features drawn exclusively from month T and earlier — a strict forward-looking definition.

Safe label pairs (T → T+1):
- Features from T ∈ {2025-01 … 2026-04}, label from T+1 ∈ {2025-02 … 2026-05}
- `2026-06` is **never used for labels** — touch it only to test query mechanics
- Per-client gate: `dim_clients.gsc_data_start ≤ first day of month T`

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Q2 — Tables:**

| Table | Role |
|---|---|
| `fact_daily_sample` | Primary source — filter to month T, aggregate per `content_hash_id` |
| `dim_content` | Static item attributes — LEFT JOIN on `content_hash_id` |
| `dim_clients` | Panel-health gate — exclude clients whose `gsc_data_start` > start of month T |
| `fact_content_query_90d` | Optional query-diversity signals — window alignment required before use |

**Q4 — Label / output (formal contract statement):**

- **Training target:** binary — `is_declining = 1` if `impressions_{T+1} < 0.8 × impressions_T`, else `0`.
- **Ranking score:** `predict_proba()[:, 1]` — the continuous probability of decline. Content items are ranked highest-to-lowest by this score.
- **Evaluation:** Precision@100 on the ranked list. The 0.8 threshold is pre-registered here and must not be tuned post-hoc to inflate the metric.

**Q5 — Excluded (one deliberate exclusion):**

| Column | Classification | Why |
|---|---|---|
| `gsc_impressions` from month T+1 | **Excluded — leakage** | This is the outcome variable itself; any column from the label window is never a feature |
| `ga4_*` where `ga4_data_available IS FALSE` | **Excluded — panel violation** | These cells are zero-filled placeholders, not real engagement data; using them teaches the model that "zero GA4 = declining," which reflects missing coverage, not content performance |
| `content_hash_id`, `client_hash_id` | **Context** | Pseudonymous IDs — grouping, joining, and client-stratified splitting only; never model inputs |

**Five features — all from month T only:**

| Feature | Source column(s) | Knowable at decision moment because… |
|---|---|---|
| `imp_T` | `SUM(gsc_impressions)` for month T | Impression data for month T is fully observed before we predict month T+1 |
| `pos_T` | `AVG(gsc_avg_position)` for month T | Average SERP position is a trailing aggregate over completed days in month T |
| `clk_T` | `SUM(gsc_clicks)` for month T | Click counts for month T are final once the month closes |
| `imp_mom_prev` | `SUM(gsc_impressions)` for month T−1 | Prior-month impressions are even older — fully in the past at decision time |
| `content_type` | `dim_content.content_type` | Static item attribute set at creation; does not change between T and T+1 |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query is a guess.*

### 3a — Grain check: `content_hash_id × month` is unique in `fact_daily_sample` for 2026-03

In [ ]:
# Grain check — zero rows back means the grain holds.
# We aggregate daily rows into one row per content_hash_id per month first,
# then probe for duplicates at that grain.
grain_check = con.sql(f"""
    WITH monthly AS (
        SELECT
            content_hash_id,
            DATE_TRUNC('month', report_date) AS month,
            SUM(gsc_impressions)             AS imp_T
        FROM {TABLES['fact_daily_sample']}
        WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
        GROUP BY 1, 2
    )
    SELECT content_hash_id, month, COUNT(*) AS c
    FROM monthly
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Duplicate grain rows: {len(grain_check)}   (expect 0 — zero means grain holds)')
grain_check

### 3b — Row count + date span for `month=2026-03`

In [ ]:
# Row count and date span — match against SKILL.md numbers
span = con.sql(f"""
    SELECT
        COUNT(*)                    AS total_daily_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)            AS earliest_date,
        MAX(report_date)            AS latest_date
    FROM {TABLES['fact_daily_sample']}
    WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

print('2026-03 slice summary:')
print(span.T.to_string())

### 3c — GA4 availability filter: before vs after applying `ga4_data_available IS TRUE`

In [ ]:
# Availability filter — show before/after row counts.
# Rows where ga4_data_available IS FALSE have zero-filled GA4 columns.
# Using them without filtering injects a synthetic signal.
before = con.sql(f"""
    SELECT COUNT(*) AS rows_before_filter
    FROM {TABLES['fact_daily_sample']}
    WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").fetchone()[0]

after = con.sql(f"""
    SELECT COUNT(*) AS rows_after_filter
    FROM {TABLES['fact_daily_sample']}
    WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
      AND ga4_data_available IS TRUE
""").fetchone()[0]

print(f'Rows BEFORE ga4_data_available IS TRUE filter : {before:>10,}')
print(f'Rows AFTER  ga4_data_available IS TRUE filter : {after:>10,}')
print(f'Rows dropped (zero-filled GA4 placeholders)   : {before - after:>10,}')
print(f'Drop rate: {(before - after) / before:.1%}')
print()
print('These dropped rows are NOT "no engagement" — they are missing data.')
print('Any GA4 feature must be built only from the rows that survive this filter.')

## 4. Five features — built from month T only

Each feature aggregated per `content_hash_id` over days in month T, then joined to the label (month T+1 outcome). No column from month T+1 is used.

In [ ]:
# Build the five features for month T = 2026-03
# and the label from month T+1 = 2026-04.
# Neither month is 2026-06 — that month is off-limits for label logic.

feature_T = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)  AS imp_T,
        AVG(gsc_avg_position) AS pos_T,
        SUM(gsc_clicks)       AS clk_T
    FROM {TABLES['fact_daily_sample']}
    WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
    GROUP BY 1, 2
""").df()

feature_T_prev = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)  AS imp_prev
    FROM {TABLES['fact_daily_sample']}
    WHERE DATE_TRUNC('month', report_date) = DATE '2026-02-01'
    GROUP BY 1, 2
""").df()

label_T1 = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS imp_T1
    FROM {TABLES['fact_daily_sample']}
    WHERE DATE_TRUNC('month', report_date) = DATE '2026-04-01'
    GROUP BY 1, 2
""").df()

content_meta = con.sql(f"""
    SELECT content_hash_id, content_type
    FROM {TABLES['dim_content']}
""").df()

import pandas as pd

# Join: features from T and T-1, label from T+1
data = (
    feature_T
    .merge(feature_T_prev, on=['content_hash_id', 'client_hash_id'], how='left', suffixes=('', '_prev'))
    .merge(label_T1,       on=['content_hash_id', 'client_hash_id'], how='inner')
    .merge(content_meta,   on='content_hash_id',                     how='left')
)

# Label: >20% impression drop month-over-month (threshold pre-registered)
data['is_declining'] = (data['imp_T1'] < 0.8 * data['imp_T']).astype(int)

print(f'Rows with both T and T+1 data: {len(data):,}')
print(f'Decline rate: {data["is_declining"].mean():.1%}')
print()
print('Feature columns:', [c for c in ['imp_T', 'pos_T', 'clk_T', 'imp_prev', 'content_type']])
data[['content_hash_id', 'imp_T', 'pos_T', 'clk_T', 'imp_prev', 'content_type', 'is_declining']].head(10)

## Part 4 — The leakage trap

Add one column derived from `impressions_{T+1}` as a feature, watch Precision@100 jump toward 1.0, then remove it.

The trap column is `imp_T1` — the very thing we are predicting. Adding it to the feature set is obvious here, but in practice leakage is subtler: a rolling average that accidentally includes T+1 data, or a join that pulls a column computed after the label window closes.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def precision_at_k(y_true, y_prob, k=100):
    top_idx = np.argsort(y_prob)[::-1][:k]
    return y_true.iloc[top_idx].mean()

clean_data = data.dropna(subset=['imp_T', 'pos_T', 'clk_T', 'imp_prev']).copy()
clean_data['content_type_code'] = clean_data['content_type'].astype('category').cat.codes

SAFE_FEATURES  = ['imp_T', 'pos_T', 'clk_T', 'imp_prev', 'content_type_code']
LEAKY_FEATURES = SAFE_FEATURES + ['imp_T1']   # ← the trap: T+1 outcome as a feature

y = clean_data['is_declining']

results = {}
for label, feats in [('Safe features (T only)', SAFE_FEATURES),
                     ('LEAKY (includes imp_T1)', LEAKY_FEATURES)]:
    X = clean_data[feats]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_te)[:, 1]
    p100  = precision_at_k(y_te.reset_index(drop=True),
                           pd.Series(probs), k=min(100, len(y_te)))
    results[label] = p100
    print(f'{label:<35}  Precision@100 = {p100:.3f}')

print()
print('=== CONCLUSION ===')
print(f'Precision@100 jumps from {results["Safe features (T only)"]:.3f} → '
      f'{results["LEAKY (includes imp_T1)"]:.3f} just by adding the outcome column.')
print('This is not a better model — it is a broken evaluation.')
print('Remove imp_T1 before any real training. The safe-feature score is the honest number.')

## Self-check

Before submitting, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere — `content_hash_id` and `client_hash_id` are pseudonyms only
- [ ] Claims use careful words: *observed*, *measured*, *directional*, *decision-support*
- [ ] 0.8 decline threshold is pre-registered in Section 2, not tuned after seeing metrics
- [ ] 2026-06 data untouched for labels
- [ ] Committed to repo under `work/notebooks/`